# Where in the world is The Turing Way? An analysis of contributor geographies

The Turing Way is a community-driven project that has attracted contributors
from around the world. In this notebook, we will analyse the geographies of our
contributors to understand where they are located and how diverse our community
is. We will use data from our contributor analytics to visualise the
distribution of contributors across different regions.

## What approach did I take to analyse the geographies of contributors?

1. **Data gathering:** The GitHub repository of The Turing Way contains
informationcabout contributors, including their GitHub profiles. We will
extract the location data from these profiles. To do this, I have used GitHub
API and a Python script to fetch the location information of each contributor
and saved it in a structured JSON format. From the JSON file, I converted the
data into a CSV file format for easier analysis.

2. **Data cleaning and geocoding:** Upon an initial manual search, I observed
that the location data from GitHub profiles can be inconsistent and may contain
various formats. For example, it could be city names, country names, city and 
country name, organisation name, continent name, and so on. Some contributors
also kept the location field empty and thus, we do not have location data for
them. We will clean the data to standardise the location information and use
geocoding services (like `geopy` from Python) to convert location names into
geographic coordinates (latitude and longitude).

3. **Data analysis and visualisation:** Once we have the geocoded data, we can
visualise the distribution of contributors across different regions. I created
an interactive world map using the library `plotly` to show the locations of
contributors.

## Python analysis

### Data gathering

The following code snippet shows how the data is gathered from the GitHub 
repository from a page that has information about all the contributors.

#### Fetching contributor data from GitHub profiles

Use the following commands in a web browser to send GitHub API requests
```
https://api.github.com/repos/the-turing-way/the-turing-way/contributors?per_page=100&page=1
https://api.github.com/repos/the-turing-way/the-turing-way/contributors?per_page=100&page=2
https://api.github.com/repos/the-turing-way/the-turing-way/contributors?per_page=100&page=3
https://api.github.com/repos/the-turing-way/the-turing-way/contributors?per_page=100&page=4
```

As a result of sending these API requests, I saved the outputs is a JSON file
format. The information from the public GitHub profiles of the contributors is
collected by this method, including their location data. This JSON file is the
basis for our analysis and visualisation of contributor geographies.

#### Converting JSON to CSV

Before going ahead with running the Python code, create a virtual environment
 with the command `python -m venv env` and activate it using `source env/bin/activate` (for Unix-based systems) or `.\env\Scripts\activate` (for Windows). Then, install the required libraries using the command `pip install requests pandas geopy plotly`.

In [ ]:
import os
import json
import requests
import pandas as pd
import time

# Read all_contributors.json

with open('data/all_contributors.json', 'r') as f:
    contributors = json.load(f)

# enter you GitHub Personal Access Token (PAT) here

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}"
}

# Gathering locations

locations = []

for contributor in contributors:
    username = contributor['login']
    profile_url = contributor["url"]

    print(f"Fetching: {username}")
    
    response = requests.get(profile_url, headers=headers) ## where has 
    
    ## the 'headers' come from?

    if response.status_code != 200:
        print(f"Failed for {username}")
        continue

    user_data = response.json()

    locations.append({
        "username": username, 
        "name": user_data.get("name"),
        "location": user_data.get("location", "Unknown"),
        "company": user_data.get("company", "Unknown"),
        "total_contributions": contributor["contributions"]
    })

# Time for API to pause and rest

time.sleep(0.2)

# Save output to CSV

df = pd.DataFrame(locations)

df.to_csv('data/contributor_locations.csv', index=False)

# To see the first 10 rows of the data frame
df.head(10);


Fetching: malvikasharan
Failed for malvikasharan
Fetching: allcontributors[bot]
Failed for allcontributors[bot]
Fetching: JimMadge
Failed for JimMadge
Fetching: EstherPlomp
Failed for EstherPlomp
Fetching: r-j-arnold
Failed for r-j-arnold
Fetching: sgibson91
Failed for sgibson91
Fetching: EKaroune
Failed for EKaroune
Fetching: aleesteele
Failed for aleesteele
Fetching: KirstieJane
Failed for KirstieJane
Fetching: da5nsy
Failed for da5nsy
Fetching: paulowoicho
Failed for paulowoicho
Fetching: BrainonSilicon
Failed for BrainonSilicon
Fetching: RichardJActon
Failed for RichardJActon
Fetching: rainsworth
Failed for rainsworth
Fetching: pherterich
Failed for pherterich
Fetching: BatoolMM
Failed for BatoolMM
Fetching: vhellon
Failed for vhellon
Fetching: Arielle-Bennett
Failed for Arielle-Bennett
Fetching: aldenc
Failed for aldenc
Fetching: jcolomb
Failed for jcolomb
Fetching: AlexandraAAJ
Failed for AlexandraAAJ
Fetching: bsipocz
Failed for bsipocz
Fetching: rosiehigman
Failed for rosiehigm

KeyboardInterrupt: 

### Data pre-processing

To display the locations on a map, we need to convert the locations from a text
or a word format to geographic coordinates (latitude and longitude). This
process is called geocoding. We will use the `geopy` library in Python to
perform geocoding. The `geopy` library provides a convenient interface to
various geocoding services, such as `Nominatim`, `Google Geocoding API`, and
others. We will use the `Nominatim` geocoding service, which is based on
OpenStreetMap data and is free to use.

In [ ]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd
import time

# Read the CSV file of all contributors and their locations

df = pd.read_csv('data/contributor_locations.csv')

# Initialize geolocator and rate limiter
geolocator = Nominatim(user_agent="contributor_analytics")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Geocode locations to get latitude and longitude
df['location'] = df['location'].fillna('Unknown')
df['latitude'] = None
df['longitude'] = None

# df.head(10)

for index, row in df.iterrows():
    location = row['location']
    if location != 'Unknown':
        try:
            geocoded_location = geocode(location)
            if geocoded_location:
                df.at[index, 'latitude'] = geocoded_location.latitude
                df.at[index, 'longitude'] = geocoded_location.longitude
                print(f"Geocoded {location}: ({geocoded_location.latitude}, {geocoded_location.longitude})")
        except Exception as e:
            print(f"Error geocoding {location}: {e}")
    time.sleep(1)  # To respect rate limits

# Save the updated DataFrame with geocoded locations
df.to_csv('data/contributor_locations_geocoded.csv', index=False)

# df.head(5)

Geocoded Aruba: (12.5013629, -69.9618475)
Geocoded UK: (54.7023545, -3.2765753)
Geocoded London, UK: (51.5074456, -0.1277653)
Geocoded Berkeley, CA: (37.8708393, -122.272863)
Geocoded London, UK: (51.5074456, -0.1277653)
Geocoded United Kingdom: (54.7023545, -3.2765753)
Geocoded London, UK: (51.5074456, -0.1277653)
Geocoded Manchester, UK: (53.4424618, -2.2324547)
Geocoded Edinburgh, UK: (55.9533456, -3.1883749)
Geocoded Liverpool, UK: (53.3933411, -2.9166389)
Geocoded berlin: (52.5173885, 13.3951309)
Geocoded United Kingdom: (54.7023545, -3.2765753)
Geocoded Seattle, WA: (47.6038321, -122.330062)
Geocoded Frankfurt am Main, Germany: (50.1106444, 8.6820917)
Geocoded London: (51.5074456, -0.1277653)
Geocoded Swansea UK: (51.6195955, -3.9459248)
Geocoded London: (51.5074456, -0.1277653)
Geocoded Melbourne, London: (51.5681666, 0.0737354)
Geocoded Estonia: (58.7523778, 25.3319078)
Geocoded Madrid, Spain: (40.416782, -3.703507)
Geocoded planet earth: (1.3165066, 103.8625991)
Geocoded Bueno

RateLimiter caught an error, retrying (0/2 tries). Called with (*('Flagstaff, Arizona, USA',), **{}).
Traceback (most recent call last):
  File "/Users/jyotibhogal/Documents/GitHub/the-turing-way/book/.venv/lib/python3.14/site-packages/urllib3/connection.py", line 204, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/jyotibhogal/Documents/GitHub/the-turing-way/book/.venv/lib/python3.14/site-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.2/Frameworks/Python.framework/Versions/3.14/lib/python3.14/socket.py", line 983, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^

Geocoded Amsterdam: (52.3730796, 4.8924534)
Geocoded Finland & Italy: (60.1612812, 24.7375492)
Geocoded Cambridge, UK: (52.1975846, 0.1391537)
Geocoded London: (51.5074456, -0.1277653)
Geocoded Manitoba, Canada: (55.001251, -97.001038)
Geocoded Swansea, UK: (51.6195955, -3.9459248)
Geocoded Swansea: (51.6195955, -3.9459248)
Geocoded Jena, Germany: (50.9281717, 11.5879359)
Geocoded London, England: (51.5074456, -0.1277653)
Geocoded Washington, DC: (38.8950982, -77.0363849)
Geocoded Cambridge: (52.1975846, 0.1391537)


### Visualisation

In [ ]:
import pandas as pd 
import plotly.express as px
import time

# Read the already geocoded CSV

df_map = pd.read_csv('data/contributor_locations_geocoded.csv')

# To view first 10 rows of the data frame of the data which is appended with 
# two new columns - latitude and longitude
# df_map.head(10)

df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# Creating a subset of the data for which the location is available by dropping
# off all rows in which either the latitude and longitude are not available.

df_plot = df_map.dropna(subset=['latitude', 'longitude'])

# Dimension of the dataframes with all the rows 
# print(df_map.shape[0])

# Dimension of the dataframes with only the rows with geocoded locations, 
# i.e., non-NA geocoded locations
# print(df_plot.shape[0])

# Create a scatter mapbox visualization
fig = px.scatter_map(df_plot,
                        lat='latitude',
                        lon='longitude',
                        hover_name='username',
                        hover_data=['name', 'company', 'total_contributions'],
                        color='total_contributions',
                        size='total_contributions',
                        color_continuous_scale=px.colors.sequential.Plasma,
                        size_max=25,
                        zoom=1, opacity=0.5)
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()
fig.write_html("contributors_world_map.html",include_plotlyjs="cdn")
